In [1]:
!pip install transformers torch nltk sentencepiece

In [2]:
# TÓM TẮT VĂN BẢN TỪ VIDEO (txt,srt,vtt)

# pip install transformers torch sentencepiece

import re
import os
from typing import List

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
# 1. CONFIG
MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"

MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 150
MIN_NEW_TOKENS = 60
CHUNK_SIZE = 250

In [4]:
# 2. LOAD MODEL
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 768)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 768)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
         

In [5]:
# 3. CLEAN SUBTITLE
def clean_subtitle_text(text: str) -> str:
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue
        if re.fullmatch(r"\d+", line):
            continue
        if re.search(r"\d{2}:\d{2}:\d{2}", line):
            continue
        if line.upper() == "WEBVTT":
            continue

        cleaned_lines.append(line)

    text = " ".join(cleaned_lines)
    text = re.sub(r"\[.*?\]", " ", text)
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
# 4. ĐỌC FILE
def read_text_file(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    if ext == ".txt":
        return content

    if ext in [".srt", ".vtt"]:
        return clean_subtitle_text(content)

    return content

In [7]:
# 5. CLEAN TEXT
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

In [8]:
# 6. SPLIT SENTENCES
def split_sentences(text: str):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

In [9]:
# 7. CHIA CHUNK
def split_text_into_chunks(text: str, max_tokens: int = CHUNK_SIZE) -> List[str]:
    sentences = split_sentences(text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        temp = current_chunk + " " + sentence

        if len(temp) < 1000:
            token_count = len(tokenizer.encode(temp, add_special_tokens=False))
        else:
            token_count = max_tokens + 1

        if token_count <= max_tokens:
            current_chunk = temp
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [10]:
# 8. TÓM TẮT 1 CHUNK
def summarize_chunk(text: str) -> str:
    text = normalize_text(text)

    inputs = tokenizer(
        text,
        max_length=MAX_INPUT_TOKENS,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            min_new_tokens=MIN_NEW_TOKENS,
            num_beams=3,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return summary.strip()

In [ ]:
# 9. TÓM TẮT VĂN BẢN DÀI
def summarize_long_text(text: str) -> str:
    text = normalize_text(text)

    token_len = len(tokenizer.encode(text, add_special_tokens=False))

    # nếu ngắn → tóm tắt luôn
    if token_len <= MAX_INPUT_TOKENS:
        return summarize_chunk(text)

    # chia đoạn
    chunks = split_text_into_chunks(text)
    print(f"Số đoạn: {len(chunks)}")

    chunk_summaries = []

    for i, chunk in enumerate(chunks, 1):
        print(f"Đang xử lý đoạn {i}/{len(chunks)}...")
        try:
            summary = summarize_chunk(chunk)
            if summary:
                chunk_summaries.append(summary)
        except Exception as e:
            print(f"Lỗi đoạn {i}: {e}")

    #  GHÉP LẠI
    final_summary = " ".join(chunk_summaries)

    # clean format
    final_summary = final_summary.replace(" .", ".")
    final_summary = final_summary.replace(" ,", ",")

    return final_summary.strip()

In [12]:
# 10. MAIN
if __name__ == "__main__":
    input_file = "output.txt"

    if not os.path.exists(input_file):
        print("Không tìm thấy file.")
        exit()

    text = read_text_file(input_file)

    if not text.strip():
        print("File không có nội dung hợp lệ.")
        exit()

    summary = summarize_long_text(text)

    print("\n=========== KẾT QUẢ ===========\n")
    print(summary)

    with open("summary_output.txt", "w", encoding="utf-8") as f:
        f.write(summary)

    print("\nĐã lưu vào summary_output.txt")

Số đoạn: 9
Đang xử lý đoạn 1/9...
Đang xử lý đoạn 2/9...
Đang xử lý đoạn 3/9...
Đang xử lý đoạn 4/9...
Đang xử lý đoạn 5/9...
Đang xử lý đoạn 6/9...
Đang xử lý đoạn 7/9...
Đang xử lý đoạn 8/9...
Đang xử lý đoạn 9/9...

=========== KẾT QUẢ ===========

Tổng bí thư chủ tịch nước tô lâm và phu nhân đã lên đường thăm cấp nhà nước tới trung quốc trong chuyến thăm đầu tiên của lãnh đạo cộng hoà xã hội chủ nghĩa việt nam. Đảng Cộng sản Việt Nam và Trung Quốc đang có chuyến thăm chính thức lần thứ hai tới Hà Nội vào ngày 10/12 năm nay trong chuyến công du lịch sử của lãnh đạo hai nước. Tổng bí thư chủ tịch nước tô lâm đã có bài viết đăng trên báo nhân dân nhật báo của trung quốc đề nghị nâng tầm kết nối chiến lược trong giai đoạn phát triển mới. Tổng bí thư chủ tịch nước Nguyễn Tấn Dũng gửi gắm tình cảm chân thành sự ưu tiên hàng đầu của đảng nhà nước và nhân dân trung quốc. Tổng bí thư chủ tịch nước tô lâm đã nhấn mạnh năm hai nghìn không trăm hai mươi sáu năm thiết lập quan hệ với Trung Quốc